# Phase 1 — Connection Test: SparkMagic → Livy → Spark

**Goal:** Verify the full chain works end-to-end with no authentication.

**Kernel to select:** `PySpark` (provided by SparkMagic's `pysparkkernel`)

When you start the kernel, SparkMagic will automatically open a Livy session.
A new Spark application will appear in the Spark UI at http://localhost:8080.

---

## 1 — Session Info
Prints Livy session ID, YARN app ID (or Spark standalone app ID), and Spark version.

In [14]:
%%info

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,None,pyspark,idle,,,None,✔


## 2 — PySpark: Spark Version and Simple RDD
Runs Python code remotely on the Spark driver via Livy.

In [15]:
print(f"Spark version : {sc.version}")
print(f"Spark master  : {sc.master}")
print(f"App name      : {sc.appName}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Spark version : 3.5.3
Spark master  : spark://spark-master:7077
App name      : livy-session-0

In [16]:
# Simple distributed computation — sum numbers 1..100 across executors
total = sc.parallelize(range(1, 5001)).sum()
print(f"Sum 1..5000 = {total}")   # expected: 5050

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Sum 1..5000 = 12502500

## 3 — Spark SQL: Basic Query
Runs SparkSQL remotely; result is rendered as a DataFrame table.

In [17]:
%%sql
SHOW DATABASES

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [18]:
# Create an in-memory DataFrame and run SQL against it
data = [("Alice", 30), ("Bob", 25), ("Charlie", 35), ("Sahbi", 22)]
df = spark.createDataFrame(data, ["name", "age"])
df.createOrReplaceTempView("people")
print("Registered temp view: people")
print(df.head(5))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Registered temp view: people
[Row(name='Alice', age=30), Row(name='Bob', age=25), Row(name='Charlie', age=35), Row(name='Sahbi', age=22)]

In [20]:
%%sql
SELECT name, age
FROM   people
WHERE  age < 28
ORDER BY age DESC

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/opt/conda/lib/python3.11/site-packages/sparkmagic/utils/utils.py:44: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[column_name] = pd.to_datetime(df[column_name], errors="raise")


Output()

## 4 — Session Management
Inspect and list sessions via the `%manage_spark` widget or via `%%info`.

In [ ]:
%manage_spark

## 5 — Verify via Livy REST directly (run in a Terminal)

Open a terminal inside JupyterLab and run:

```bash
# List all active sessions
curl -s http://livy:8998/sessions | python3 -m json.tool

# Get session logs (replace 0 with the session ID shown in %%info)
curl -s http://livy:8998/sessions/0/log | python3 -m json.tool
```

Or from your host machine:
```bash
curl -s http://localhost:8998/sessions | python3 -m json.tool
```